# Tutorial: Heterogeneous Timescales (Rossler-Lorenz)

Many real coupled systems have **different intrinsic frequencies**. For example:
- Brain oscillations: theta (~6 Hz) coupled with gamma (~40 Hz)
- Climate: ocean currents (years) coupled with atmospheric dynamics (days)
- The Rossler oscillator (~0.06 Hz) coupled with the Lorenz system (~1 Hz)

A naive Takens embedding uses the **same delay** for both signals, which is wrong when their timescales differ. ATT's `JointEmbedder` estimates **per-channel delays**, which is critical for correct binding detection.

This tutorial demonstrates:
1. Why shared delays produce degenerate embeddings
2. How per-channel delays fix this
3. End-to-end binding detection on a heterogeneous system

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from att.config import set_seed
from att.synthetic import coupled_rossler_lorenz
from att.embedding import TakensEmbedder, JointEmbedder, validate_embedding
from att.binding import BindingDetector

set_seed(42)
print("ATT loaded.")

## Step 1: Generate the coupled system

The **coupled Rossler-Lorenz** system pairs a slow Rossler oscillator with a fast Lorenz oscillator. The Rossler has a natural period of ~17 time units; the Lorenz oscillates ~16x faster. They are coupled diffusively through their x-components.

In [ ]:
coupling = 0.3
n_steps = 8000
transient = 1000  # discard initial transient

ts_r, ts_l = coupled_rossler_lorenz(n_steps=n_steps, coupling=coupling, seed=42)
X = ts_r[transient:, 0]  # Rossler x-component (slow)
Y = ts_l[transient:, 0]  # Lorenz x-component (fast)

fig, axes = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
axes[0].plot(X[:2000], linewidth=0.5, color="C0")
axes[0].set_ylabel("Rossler x(t)")
axes[0].set_title(f"Coupled Rossler-Lorenz (coupling = {coupling})")
axes[1].plot(Y[:2000], linewidth=0.5, color="C1")
axes[1].set_ylabel("Lorenz x(t)")
axes[1].set_xlabel("Time step")
plt.tight_layout()
plt.show()

print("Notice the different frequencies: Rossler is much slower than Lorenz.")

## Step 2: The shared-delay problem

If we use the **same delay** for both channels (say $\tau=10$), it's a bad compromise:
- Too large for the fast Lorenz signal (oversamples the attractor)
- Too small for the slow Rossler signal (undersamples the attractor)

This produces a **degenerate** joint embedding with a very high condition number, meaning the point cloud is nearly flat in some directions.

In [ ]:
# Shared delay: tau=10 for both channels, dimension=3
shared_delay = 10

emb_x_shared = TakensEmbedder(delay=shared_delay, dimension=3)
emb_y_shared = TakensEmbedder(delay=shared_delay, dimension=3)
joint_shared = JointEmbedder(delays=[shared_delay, shared_delay], dimensions=[3, 3])

cloud_x_shared = emb_x_shared.fit_transform(X)
cloud_y_shared = emb_y_shared.fit_transform(Y)
cloud_joint_shared = joint_shared.fit_transform([X, Y])

q_x = validate_embedding(cloud_x_shared)
q_y = validate_embedding(cloud_y_shared)
q_j = validate_embedding(cloud_joint_shared)

print("=== Shared delay (tau=10) ===")
print(f"  Rossler marginal:  cond = {q_x['condition_number']:8.1f}  degenerate = {q_x['degenerate']}")
print(f"  Lorenz marginal:   cond = {q_y['condition_number']:8.1f}  degenerate = {q_y['degenerate']}")
print(f"  Joint embedding:   cond = {q_j['condition_number']:8.1f}  degenerate = {q_j['degenerate']}")
print(f"\nThe joint condition number ({q_j['condition_number']:.0f}) is very high!")
print("This means the 6D point cloud is nearly flat --- topology will be unreliable.")

## Step 3: Per-channel delays fix this

ATT's `TakensEmbedder(delay='auto')` estimates the optimal delay for each signal independently using AMI (Average Mutual Information). The Rossler needs a larger delay; the Lorenz needs a smaller one.

In [ ]:
# Auto delays: each channel gets its own optimal delay
emb_x_auto = TakensEmbedder(delay="auto", dimension="auto")
emb_y_auto = TakensEmbedder(delay="auto", dimension="auto")

cloud_x_auto = emb_x_auto.fit_transform(X)
cloud_y_auto = emb_y_auto.fit_transform(Y)

print(f"Rossler: delay = {emb_x_auto.delay_}, dimension = {emb_x_auto.dimension_}")
print(f"Lorenz:  delay = {emb_y_auto.delay_}, dimension = {emb_y_auto.dimension_}")

q_x_auto = validate_embedding(cloud_x_auto)
q_y_auto = validate_embedding(cloud_y_auto)

print("\n=== Per-channel delays (auto) ===")
print(f"  Rossler marginal:  cond = {q_x_auto['condition_number']:8.1f}")
print(f"  Lorenz marginal:   cond = {q_y_auto['condition_number']:8.1f}")
print(f"\nCondition numbers are {q_j['condition_number']/max(q_x_auto['condition_number'], q_y_auto['condition_number']):.0f}x better than shared delay!")

## Step 4: Binding detection --- shared vs auto

Let's run binding detection both ways and compare. The auto-delay version should produce more reliable results.

In [ ]:
subsample = 500

# Auto delays
det_auto = BindingDetector(max_dim=1, baseline="max", embedding_quality_gate=False)
det_auto.fit(X, Y, subsample=subsample, seed=42)
score_auto = det_auto.binding_score()
eq_auto = det_auto.embedding_quality()

# Shared delay
det_shared = BindingDetector(max_dim=1, baseline="max", embedding_quality_gate=False)
det_shared.fit(
    X, Y,
    joint_embedder=joint_shared,
    marginal_embedder_x=emb_x_shared,
    marginal_embedder_y=emb_y_shared,
    subsample=subsample, seed=42,
)
score_shared = det_shared.binding_score()
eq_shared = det_shared.embedding_quality()

print(f"Binding score (auto delays):   {score_auto:.2f}  (joint cond = {eq_auto['joint']['condition_number']:.0f})")
print(f"Binding score (shared delay):  {score_shared:.2f}  (joint cond = {eq_shared['joint']['condition_number']:.0f})")

## Step 5: The embedding quality gate

ATT includes an **embedding quality gate** that warns you when the joint embedding is degenerate. When `embedding_quality_gate=True` (the default), the detector will raise a warning if the condition number exceeds the threshold.

In [ ]:
import warnings

# With the quality gate ON and shared delay --- expect a warning
det_gated = BindingDetector(max_dim=1, baseline="max", embedding_quality_gate=True)

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    det_gated.fit(
        X, Y,
        joint_embedder=joint_shared,
        marginal_embedder_x=emb_x_shared,
        marginal_embedder_y=emb_y_shared,
        subsample=subsample, seed=42,
    )
    if w:
        print(f"Quality gate fired! Warning: {w[0].message}")
    else:
        print("No warning --- embedding quality is acceptable.")

print("\nWith auto delays and quality gate ON, no warning is expected:")
det_gated2 = BindingDetector(max_dim=1, baseline="max", embedding_quality_gate=True)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    det_gated2.fit(X, Y, subsample=subsample, seed=42)
    if w:
        print(f"Warning: {w[0].message}")
    else:
        print("No warning --- auto delays produce a healthy embedding.")

## Step 6: Coupling sweep

Let's sweep coupling strength and see how binding score responds with each approach:

In [ ]:
coupling_values = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8]
scores_auto_sweep = []
scores_shared_sweep = []

for c in coupling_values:
    ts_r, ts_l = coupled_rossler_lorenz(n_steps=n_steps, coupling=c, seed=42)
    Xc = ts_r[transient:, 0]
    Yc = ts_l[transient:, 0]

    # Auto
    d = BindingDetector(max_dim=1, baseline="max", embedding_quality_gate=False)
    d.fit(Xc, Yc, subsample=subsample, seed=42)
    scores_auto_sweep.append(d.binding_score())

    # Shared
    ex = TakensEmbedder(delay=10, dimension=3)
    ey = TakensEmbedder(delay=10, dimension=3)
    ej = JointEmbedder(delays=[10, 10], dimensions=[3, 3])
    d2 = BindingDetector(max_dim=1, baseline="max", embedding_quality_gate=False)
    d2.fit(Xc, Yc, joint_embedder=ej, marginal_embedder_x=ex, marginal_embedder_y=ey,
           subsample=subsample, seed=42)
    scores_shared_sweep.append(d2.binding_score())

    print(f"  c={c:.2f}: auto={scores_auto_sweep[-1]:.1f}, shared={scores_shared_sweep[-1]:.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(coupling_values, scores_auto_sweep, "o-", color="#1976D2", linewidth=2,
        markersize=7, label="Per-channel delays (auto)")
ax.plot(coupling_values, scores_shared_sweep, "s--", color="#E64A19", linewidth=2,
        markersize=7, label="Shared delay (tau=10)")
ax.set_xlabel("Coupling strength")
ax.set_ylabel("Binding score")
ax.set_title("Rossler-Lorenz: per-channel vs shared delays")
ax.legend(frameon=True)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Per-channel delays produce a more interpretable coupling response.")

## Key takeaways

1. **Shared delays are wrong** for heterogeneous-timescale systems. The joint embedding becomes degenerate (high condition number), and binding scores become unreliable.

2. **Per-channel delay estimation** (AMI per signal) is critical. ATT's `BindingDetector` does this automatically when you don't pass custom embedders.

3. **The embedding quality gate** catches degenerate embeddings before they produce misleading topology. Always leave it enabled (the default).

4. **This matters for neuroscience**: EEG channels in different frequency bands (theta, alpha, gamma) have different optimal embedding delays. ATT handles this automatically.

## What next?

- **Lorenz walkthrough**: See `tutorial_lorenz_walkthrough.ipynb` for the full pipeline on a single-timescale system
- **Real EEG**: See `fig7_real_eeg.ipynb` for how per-channel embedding works on neural data
- **Z-score calibration**: See `fig9_zscore_calibration.py` for how to calibrate binding scores across systems
- **API docs**: https://musicofhel.github.io/att-docs/